# Cost Analysis: Conservative vs E-Reserve Energy Models




## 1. Load Required Packages and Set Configuration

In [ ]:
# Load packages
import pandas as pd
import matplotlib.pyplot as plt
import os,sys
import plotly.express as px
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import read_KPI_adequacy

# Set visualization dimensions
dim = (1000, 500)

## 2. Define Solution Scenarios and Load KPI Data

In [ ]:
ss = [
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
]

gcd_KPI_adequacy, gcdi_KPI_adequacy = read_KPI_adequacy(ss)

## 3. Data Filtering and Preparation

In [ ]:
# Create filter for days with data for both model types
filter_ = gcd_KPI_adequacy.pivot(
    index='day',
    columns='model_type',
    values='solution_id'
).dropna().index

print(f"Number of valid days: {len(filter_)}")
print(f"Days range: {filter_.min()} to {filter_.max()}")

# gcd_KPI_adequacy = gcd_KPI_adequacy[gcd_KPI_adequacy['day'].isin(filter_)]

## 4. Cost Difference Analysis Between Model Types

In [ ]:
# Group by day and calculate differences between model types
cost_comparison = gcd_KPI_adequacy.pivot_table(
    index='day',
    columns='model_type',
    values='OV_uc',
    aggfunc='first'
).dropna() # drops days without both model types

# Calculate the difference: envelope - e-reserve
cost_comparison['OV_uc_diff'] = cost_comparison['envelope'] - cost_comparison['e-reserve']
cost_comparison['OV_uc_diff_pct'] = (cost_comparison['envelope'] - cost_comparison['e-reserve']) / cost_comparison['e-reserve'] * 100

print("Cost comparison summary:")
print(f"Mean difference: {cost_comparison['OV_uc_diff'].mean():.2f}")
print(f"Mean percentage difference: {cost_comparison['OV_uc_diff_pct'].mean():.2f}%")
cost_comparison.head()

## 5. Prepare Data for Analysis

In [ ]:
# Get all cost columns for PCA analysis
cost_columns = [col for col in gcd_KPI_adequacy.columns if 'cost' in col and '_uc' in col]
print("Cost columns:", cost_columns)

# Create a comprehensive dataset with all cost data
cost_data = gcd_KPI_adequacy.pivot_table(
    index='day',
    columns='model_type', 
    values=['OV_uc'] + cost_columns,
    aggfunc='first'
).dropna()

# Flatten column names for easier handling
cost_data.columns = ['_'.join(col) for col in cost_data.columns]
print(f"Cost dataset shape: {cost_data.shape}")
cost_data.head()

## 6. Comprehensive Cost Component Analysis

Let's analyze what drives the cost differences between envelope and e-reserve models by examining individual cost components.

In [ ]:
# Calculate differences for all cost components
cost_differences = pd.DataFrame(index=filter_)

for component in ['OV_uc'] + cost_columns:
    if component in gcd_KPI_adequacy.columns:
        pivot_comp = gcd_KPI_adequacy.pivot_table(
            index='day',
            columns='model_type',
            values=component,
            aggfunc='first'
        ).dropna()
        
        if 'envelope' in pivot_comp.columns and 'e-reserve' in pivot_comp.columns:
            cost_differences[f'{component}_diff'] = pivot_comp['envelope'] - pivot_comp['e-reserve']
            cost_differences[f'{component}_diff_pct'] = ((pivot_comp['envelope'] - pivot_comp['e-reserve']) / 
                                                        pivot_comp['e-reserve'] * 100)

print(f"Cost differences dataframe shape: {cost_differences.shape}")
print(f"Available difference columns: {list(cost_differences.columns)}")
cost_differences.head()

## 7. Correlation Analysis - What Drives OV_uc Differences?

In [ ]:
# Calculate correlations between OV_uc differences and component differences
correlations_abs = cost_differences.corr()['OV_uc_diff'].drop('OV_uc_diff').sort_values(ascending=False, key=abs)
correlations_pct = cost_differences.corr()['OV_uc_diff_pct'].drop('OV_uc_diff_pct').sort_values(ascending=False, key=abs)

print("=== CORRELATION ANALYSIS ===")
print("\nTop correlations with OV_uc absolute difference:")
print(correlations_abs.head(10))

print("\nTop correlations with OV_uc percentage difference:")
print(correlations_pct.head(10))

# Focus on suspected thermal components
suspected_components = ['thermal_production_cost_uc', 'thermal_start_cost_uc']
print(f"\n=== SUSPECTED COMPONENTS ANALYSIS ===")

for component in suspected_components:
    diff_col = f'{component}_diff'
    if diff_col in correlations_abs.index:
        corr_abs = correlations_abs[diff_col]
        corr_pct = correlations_pct.get(f'{component}_diff_pct', 'N/A')
        print(f"\n{component}:")
        print(f"  Correlation with OV_uc_diff: {corr_abs:.4f}")
        print(f"  Correlation with OV_uc_diff_pct: {corr_pct:.4f}" if corr_pct != 'N/A' else f"  Percentage correlation: N/A")
    else:
        print(f"\n{component}: Not found in data")

## 8. Component Contribution Analysis

In [ ]:
# Calculate what percentage of total difference each component explains
component_contributions = pd.DataFrame()

for component in cost_columns:
    diff_col = f'{component}_diff'
    if diff_col in cost_differences.columns:
        # Calculate absolute contribution
        abs_contrib = cost_differences[diff_col].abs().sum() / cost_differences['OV_uc_diff'].abs().sum() * 100
        
        # Calculate mean difference
        mean_diff = cost_differences[diff_col].mean()
        
        # Calculate correlation with total difference
        correlation = cost_differences[diff_col].corr(cost_differences['OV_uc_diff'])
        
        component_contributions = pd.concat([
            component_contributions,
            pd.DataFrame({
                'component': [component],
                'mean_difference': [mean_diff],
                'abs_contribution_pct': [abs_contrib],
                'correlation_with_total': [correlation],
                'std_difference': [cost_differences[diff_col].std()]
            })
        ])

component_contributions = component_contributions.sort_values('abs_contribution_pct', ascending=False).reset_index(drop=True)

print("=== COMPONENT CONTRIBUTION ANALYSIS ===")
print("\nComponents ranked by their contribution to total cost difference:")
print(component_contributions.round(4))

## 9. Visualize Component Contributions

In [ ]:
# Create interactive bar chart of component contributions
fig_contrib = px.bar(
    component_contributions.head(10),
    x='abs_contribution_pct',
    y='component',
    orientation='h',
    title='Component Contributions to Total Cost Difference (%)',
    labels={'abs_contribution_pct': 'Contribution (%)', 'component': 'Cost Component'},
    color='correlation_with_total',
    color_continuous_scale='RdBu'
)

fig_contrib.update_layout(
    height=600,
    width=1000,
    yaxis={'categoryorder': 'total ascending'}
)

fig_contrib.show()

## 10. Time Series Analysis of Key Components

In [ ]:
# Plot time series of the top 5 contributing components
from plotly.subplots import make_subplots
import plotly.graph_objects as go

top_5_components = component_contributions.head(5)['component'].tolist()

# Create subplot for time series
fig = make_subplots(
    rows=len(top_5_components), 
    cols=1,
    subplot_titles=[f"{comp} Differences Over Time" for comp in top_5_components],
    shared_xaxes=True
)

for i, component in enumerate(top_5_components, 1):
    diff_col = f'{component}_diff'
    if diff_col in cost_differences.columns:
        fig.add_trace(
            go.Scatter(
                x=cost_differences.index,
                y=cost_differences[diff_col],
                name=component,
                line=dict(width=2)
            ),
            row=i, col=1
        )

fig.update_layout(
    height=800,
    width=1200,
    title_text="Time Series of Top 5 Cost Component Differences"
)

fig.show()

## 11. Scatter Plot Analysis - Relationship Between Components

In [ ]:
# Create scatter plots for suspected components vs total cost difference
fig_scatter = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Thermal Production Cost vs Total', 'Thermal Start Cost vs Total', 'LGEN_cost_uc']
)

suspected_components = ['thermal_production_cost_uc', 'thermal_start_cost_uc', 'LGEN_cost_uc']

for i, component in enumerate(suspected_components, 1):
    diff_col = f'{component}_diff'
    if diff_col in cost_differences.columns:
        fig_scatter.add_trace(
            go.Scatter(
                x=cost_differences[diff_col],
                y=cost_differences['OV_uc_diff'],
                mode='markers',
                name=component,
                text=cost_differences.index,  # Show day number on hover
                hovertemplate=f'Day: %{{text}}<br>{component}: %{{x}}<br>Total Diff: %{{y}}<extra></extra>'
            ),
            row=1, col=i
        )

fig_scatter.update_xaxes(title_text="Component Difference", row=1, col=1)
fig_scatter.update_xaxes(title_text="Component Difference", row=1, col=2)
fig_scatter.update_xaxes(title_text="Component Difference", row=1, col=3)
fig_scatter.update_yaxes(title_text="Total OV_uc Difference", row=1, col=1)

fig_scatter.update_layout(
    height=500,
    width=1200,
    title_text="Suspected Components vs Total Cost Difference"
)

fig_scatter.show()

In [ ]:
fig = px.box(cost_differences, y='OV_uc_diff_pct', title='Distribution of Total Cost Difference Percentages')
fig.update_layout(
    yaxis_title='cost difference [%]',
    width=dim[0]/2,
    height=dim[1])
fig.update_traces(
    boxmean=True,
)
stats_text = (f"Mean: {cost_differences['OV_uc_diff_pct'].mean():.2f}%\n"
              f"Median: {cost_differences['OV_uc_diff_pct'].median():.2f}%\n"
              f"Min: {cost_differences['OV_uc_diff_pct'].quantile(0):.2f}%\n"
              f"Max: {cost_differences['OV_uc_diff_pct'].quantile(1):.2f}%"
            #   f"25th percentile: {economic_data['OV_uc_%_e-reserve'].quantile(0.25):.2f}%\n"
            #   f"75th percentile: {economic_data['OV_uc_%_e-reserve'].quantile(0.75):.2f}%")
)
fig.add_annotation(
    text=stats_text,
    xref="paper", yref="paper",
    x=1., y=-0.1,
    showarrow=False,
    font=dict(size=12)
)

## 12. Statistical Decomposition Analysis

In [ ]:
# Calculate R-squared to see how much variance each component explains
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("=== STATISTICAL DECOMPOSITION OF COST DIFFERENCES ===")

r2_scores = {}
for component in cost_columns:
    diff_col = f'{component}_diff'
    if diff_col in cost_differences.columns:
        X = cost_differences[diff_col].values.reshape(-1, 1)
        y = cost_differences['OV_uc_diff'].values
        
        # Remove any NaN values
        mask = ~(np.isnan(X.flatten()) | np.isnan(y))
        if mask.sum() > 5:  # Need at least 5 points
            X_clean = X[mask].reshape(-1, 1)
            y_clean = y[mask]
            
            model = LinearRegression()
            model.fit(X_clean, y_clean)
            y_pred = model.predict(X_clean)
            r2 = r2_score(y_clean, y_pred)
            r2_scores[component] = r2

# Sort by R-squared
r2_df = pd.DataFrame(list(r2_scores.items()), columns=['component', 'r2_score'])
r2_df = r2_df.sort_values('r2_score', ascending=False)

print("\nVariance Explained (R²) by each component:")
print(r2_df.head(10))

# Multiple regression with top components
top_components = r2_df.head(3)['component'].tolist()
if len(top_components) >= 2:
    print(f"\n=== MULTIPLE REGRESSION ANALYSIS ===")
    X_multi = cost_differences[[f'{comp}_diff' for comp in top_components if f'{comp}_diff' in cost_differences.columns]]
    y_multi = cost_differences['OV_uc_diff']
    
    # Remove rows with any NaN
    mask_multi = ~(X_multi.isna().any(axis=1) | y_multi.isna())
    X_multi_clean = X_multi[mask_multi]
    y_multi_clean = y_multi[mask_multi]
    
    if len(X_multi_clean) > len(top_components):
        model_multi = LinearRegression()
        model_multi.fit(X_multi_clean, y_multi_clean)
        r2_multi = model_multi.score(X_multi_clean, y_multi_clean)
        
        print(f"Combined R² of top {len(top_components)} components: {r2_multi:.4f}")
        print("Component coefficients:")
        for comp, coef in zip(X_multi_clean.columns, model_multi.coef_):
            print(f"  {comp}: {coef:.4f}")

## 13. Summary Report

In [ ]:
print("=== COMPREHENSIVE COST DIFFERENCE ANALYSIS SUMMARY ===")

# Overall statistics
total_days = len(cost_differences)
mean_total_diff = cost_differences['OV_uc_diff'].mean()
std_total_diff = cost_differences['OV_uc_diff'].std()

print(f"\nOverall Statistics:")
print(f"  Total days analyzed: {total_days}")
print(f"  Mean OV_uc difference (envelope - e-reserve): {mean_total_diff:,.2f}")
print(f"  Standard deviation: {std_total_diff:,.2f}")
print(f"  Range: [{cost_differences['OV_uc_diff'].min():,.2f}, {cost_differences['OV_uc_diff'].max():,.2f}]")

# Top 3 drivers
if len(component_contributions) > 0:
    print(f"\nTop 3 Cost Drivers (by contribution %):")
    for i, row in component_contributions.head(3).iterrows():
        print(f"  {i+1}. {row['component']}: {row['abs_contribution_pct']:.1f}% contribution")
        print(f"     Mean difference: {row['mean_difference']:,.2f}")
        print(f"     Correlation with total: {row['correlation_with_total']:.3f}")

# Suspected components analysis
print(f"\n=== YOUR SUSPECTED COMPONENTS ===")
for component in ['thermal_production_cost_uc', 'thermal_start_cost_uc']:
    if component in component_contributions['component'].values:
        row = component_contributions[component_contributions['component'] == component].iloc[0]
        print(f"\n{component}:")
        print(f"  Contribution: {row['abs_contribution_pct']:.1f}% of total difference")
        print(f"  Correlation: {row['correlation_with_total']:.3f}")
        print(f"  Mean difference: {row['mean_difference']:,.2f}")
        if component in r2_scores:
            print(f"  Variance explained (R²): {r2_scores[component]:.3f}")
    else:
        print(f"\n{component}: Data not available in analysis")

# Key findings
print(f"\n=== KEY FINDINGS ===")
if len(component_contributions) > 0:
    top_driver = component_contributions.iloc[0]
    print(f"• Primary cost driver: {top_driver['component']} ({top_driver['abs_contribution_pct']:.1f}% contribution)")
    
    thermal_components = component_contributions[component_contributions['component'].str.contains('thermal', case=False)]
    if len(thermal_components) > 0:
        thermal_total = thermal_components['abs_contribution_pct'].sum()
        print(f"• Total thermal cost contribution: {thermal_total:.1f}%")
    
    high_corr_components = component_contributions[component_contributions['correlation_with_total'].abs() > 0.8]
    if len(high_corr_components) > 0:
        print(f"• Components with high correlation (>0.8): {len(high_corr_components)}")
        for _, row in high_corr_components.iterrows():
            print(f"  - {row['component']}: {row['correlation_with_total']:.3f}")